# **Model Evaluation**
Here we will evaluate the given, optimised XGBoost model more closely to get additional information and insights.
We will use the results of our HyperParameter selection and try to assure that our models meets our requirements.

### Requirements:
- Perform better than RandomGuessing
- Perform better than ZeroGuessing
- Respond to a new example in less than one second 
- Model should at least identify 75% fraud payments (Recall >= 0.75)
- Model should make maximal 50% false alarms (Precision >50)


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import time
import random
import statistics
import joblib
from sklearn.pipeline import Pipeline
sys.path.append('..')  
from src.data.loader import Data
from src.models.random_guessing import RandomGuessing
from src.models.zero_guessing import ZeroGuessing
from src.evaluation.evaluation_metrics import EvaluationMetrics

data = Data()
X_train, X_test, y_train, y_test= data.prepare_data()

0.8533621097323564
0


/Users/tomseidel/Desktop/Learning/ML/fraud-detection/notebooks/../src/data/data.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_day"] = df["TransactionDT"] // 86400
/Users/tomseidel/Desktop/Learning/ML/fraud-detection/notebooks/../src/data/data.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_hour"] = (df["TransactionDT"] // 3600) % 24
/Users/tomseidel/Desktop/Learning/ML/fraud-detection/notebooks/../src/data/data.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of cal

## **Model Training**
In this section we will train the model on out train data.

We will select the Hyper Paramaters that we identifies best during Hyper Parameter selection

In [2]:
model_path = Path.cwd().parent / "src" / "models" / "xgb" / "pipeline.pkl"
pipeline:Pipeline = joblib.load(filename=model_path)

## **Benchmarks (Random and Zero guessing)**
In this section we will compare the models performance against our two Base Models.


We expect the model to perform significantly better than both the Base Models.

We compare the models based on their ROC-AUC score

In [3]:
model = ZeroGuessing()
y_pred_test = model.predict_many(X=X_test)
zero_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
zero_guessing_metrics.roc_auc

/Users/tomseidel/Desktop/Learning/ML/fraud-detection/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


0.5

In [4]:
model = RandomGuessing()
y_pred_test = model.predict_many(X=X_test)
random_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
random_guessing_metrics.roc_auc

0.4961492485275593

In [5]:
y_pred_test = pipeline.predict_proba(X_test)[:, 1]
xgb_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test)
xgb_guessing_metrics.roc_auc

0.9048693223437869

In [10]:
score_dataframe = pd.DataFrame([
    {"model": "Zero Guessing",   "roc_auc": zero_guessing_metrics.roc_auc},
    {"model": "Random Guessing", "roc_auc": random_guessing_metrics.roc_auc},
    {"model": "XGBoost",         "roc_auc": xgb_guessing_metrics.roc_auc},
])
score_dataframe[score_dataframe["model"].isin(["Zero Guessing", "Random Guessing", "XGBoost"])]

,model,roc_auc
0,Zero Guessing,0.500000
1,Random Guessing,0.506348
2,XGBoost,0.963346


as expected, we can see that our optimised model performs way better than both our Benchmarks.

## **Time Measurement**
In this section we will check the response time of our trained model. We expect and request it to respond in less than one second.

We will do this by predict 100 examples sequencially and then get mean, min and max duration

In [12]:
durations = []

for i in range(100):
    start_time = time.time()
    sample = X_test.iloc[[random.randint(0, len(X_test)-1)]]  
    pipeline.predict_proba(sample)[:, 1]
    end_time = time.time()
    durations.append(end_time - start_time)
    
print("Maximum:",round(max(durations),2),"s")
print("Minimum:",round(min(durations),2),"s")
print("Average:",round(statistics.mean(durations),2),"s")

Maximum: 0.03 s
Minimum: 0.01 s
Average: 0.01 s


The model is running faster tha expected and meets our requirements

## **Precision and Recall**
In this section we want to make sure, that our model meets our given requirements for Recall and Precision which we defined at the top.

We are using the optimised treshold we found in Treshold Tuning (0.075)

In [13]:
y_pred_test = pipeline.predict_proba(X_test)[:, 1]
xgb_guessing_metrics = EvaluationMetrics(y=y_test, y_pred=y_pred_test, treshold=0.075)
xgb_guessing_metrics.print_eval_report(headline="XGBmodel")



XGBmodel
- Precision: 0.595010491956167
- Recall: 0.8234914488544692
- F1: 0.6908500270709258
- ROC-AUC: 0.9633460968241898


The model meets our requirments 

## Conclusion
All our requirements have been fullfilled and the model is ready to be used